# Modelado Predictivo MSI

Este notebook entrena y compara tres familias de modelos para predecir:

`target_msi = log(1 + fatalities)`

Diseno experimental:

- **Modelo 1:** variables estructuradas.
- **Modelo 2:** variables estructuradas + scores expertos de severidad militar.
- **Modelo 3:** Modelo 2 + componentes semanticos del texto (`emb_pca_*`).

Evaluacion principal:

- Train: eventos historicos 2024-2025.
- Test: eventos 2026.

Se reportan variantes con y sin `source` para auditar sesgo de cobertura.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = .25

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'data' / 'processed'

TARGET = 'target_msi'
AUDIT_EXCLUDE = {'event_id', 'event_date', 'fatalities', 'target_msi', 'split_recommended', 'text_clean'}

## 1. Carga de datasets

In [ ]:
paths = {
    'M1_structured': DATA_DIR / 'model1_structured_dataset.csv',
    'M2_scores': DATA_DIR / 'model2_military_scores_dataset.csv',
    'M3_embeddings': DATA_DIR / 'model3_embeddings_dataset.csv',
}

datasets = {name: pd.read_csv(path, parse_dates=['event_date']) for name, path in paths.items()}
for name, data in datasets.items():
    print(name, data.shape)
    display(data['split_recommended'].value_counts().to_frame('rows'))
    print('emb cols:', len([c for c in data.columns if c.startswith('emb_pca_')]))

display(datasets['M3_embeddings'].head(3))

## 2. Utilidades de entrenamiento

Usamos tres algoritmos por dataset:

- Ridge: baseline lineal regularizado.
- Random Forest: no lineal, robusto para interacciones.
- HistGradientBoosting: boosting eficiente para tabular.

La comparacion no busca aun hiperparametrizacion fina; busca medir ganancia incremental entre bloques de variables.

In [ ]:
def split_xy(data, include_source=True):
    drop = set(AUDIT_EXCLUDE)
    if not include_source:
        drop.add('source')
    feature_cols = [c for c in data.columns if c not in drop]
    X = data[feature_cols].copy()
    y = data[TARGET].astype(float)
    train_mask = data['split_recommended'].eq('train_historical')
    test_mask = data['split_recommended'].eq('test_2026')
    return X[train_mask], X[test_mask], y[train_mask], y[test_mask], feature_cols

def make_preprocessor(X):
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=5)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numeric_features),
        ('cat', categorical_pipe, categorical_features),
    ]), numeric_features, categorical_features

def get_models():
    return {
        'Ridge': Ridge(alpha=3.0),
        'RandomForest': RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
        'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, l2_regularization=0.1, random_state=42),
    }

def evaluate(y_true, pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, pred)))
    mae = mean_absolute_error(y_true, pred)
    r2 = r2_score(y_true, pred)
    fatal_pred = np.expm1(np.maximum(pred, 0))
    fatal_true = np.expm1(y_true)
    fatal_mae = mean_absolute_error(fatal_true, fatal_pred)
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'fatalities_mae_backtransform': fatal_mae}

def train_suite(datasets):
    rows = []
    predictions = {}
    fitted = {}
    for dataset_name, data in datasets.items():
        for include_source in [True, False]:
            X_train, X_test, y_train, y_test, feature_cols = split_xy(data, include_source=include_source)
            preprocessor, numeric_features, categorical_features = make_preprocessor(X_train)
            for model_name, model in get_models().items():
                pipe = Pipeline([
                    ('prep', preprocessor),
                    ('model', model),
                ])
                pipe.fit(X_train, y_train)
                pred = pipe.predict(X_test)
                pred = np.maximum(pred, 0)
                metrics = evaluate(y_test, pred)
                key = (dataset_name, 'with_source' if include_source else 'without_source', model_name)
                rows.append({
                    'dataset': dataset_name,
                    'source_variant': key[1],
                    'algorithm': model_name,
                    'n_train': len(X_train),
                    'n_test': len(X_test),
                    'n_features_raw': len(feature_cols),
                    **metrics,
                })
                predictions[key] = pd.DataFrame({'y_true': y_test.values, 'y_pred': pred}, index=y_test.index)
                fitted[key] = pipe
    return pd.DataFrame(rows).sort_values(['rmse', 'mae']), predictions, fitted

results, predictions, fitted = train_suite(datasets)
display(results)

## 3. Comparacion de modelos

In [ ]:
best = results.iloc[0]
print('Mejor configuracion:')
display(best.to_frame().T)

pivot = results.pivot_table(index=['dataset', 'source_variant'], columns='algorithm', values='rmse')
display(pivot)

fig, ax = plt.subplots(figsize=(12, 5))
plot_df = results.copy()
plot_df['config'] = plot_df['dataset'] + '\n' + plot_df['source_variant'] + '\n' + plot_df['algorithm']
plot_df.sort_values('rmse').plot(kind='bar', x='config', y='rmse', ax=ax, legend=False)
ax.set_title('RMSE por configuracion')
ax.set_ylabel('RMSE target_msi')
plt.xticks(rotation=80, ha='right')
plt.tight_layout()
plt.show()

## 4. Real vs predicho

In [ ]:
best_key = (best['dataset'], best['source_variant'], best['algorithm'])
pred_df = predictions[best_key]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(pred_df['y_true'], pred_df['y_pred'], alpha=.45, s=15)
limit = max(pred_df['y_true'].max(), pred_df['y_pred'].max())
axes[0].plot([0, limit], [0, limit], color='red', linestyle='--')
axes[0].set_title('Real vs predicho - MSI')
axes[0].set_xlabel('MSI real')
axes[0].set_ylabel('MSI predicho')

resid = pred_df['y_true'] - pred_df['y_pred']
axes[1].hist(resid, bins=40)
axes[1].set_title('Distribucion de residuales')
axes[1].set_xlabel('real - predicho')
plt.tight_layout()
plt.show()

display(pred_df.describe())

## 5. Errores por fuente, arma y objetivo en 2026

In [ ]:
data_for_best = datasets[best['dataset']].copy()
test_rows = data_for_best[data_for_best['split_recommended'].eq('test_2026')].copy()
test_rows = test_rows.loc[pred_df.index].copy()
test_rows['y_true'] = pred_df['y_true']
test_rows['y_pred'] = pred_df['y_pred']
test_rows['abs_error'] = (test_rows['y_true'] - test_rows['y_pred']).abs()

for col in ['source', 'weapon_type', 'target_type']:
    if col in test_rows.columns:
        err = test_rows.groupby(col).agg(
            events=('event_id', 'count'),
            mae=('abs_error', 'mean'),
            y_true_mean=('y_true', 'mean'),
            y_pred_mean=('y_pred', 'mean'),
        ).sort_values('mae', ascending=False)
        print('\n###', col)
        display(err)

## 6. Importancia de variables

Para Random Forest se reporta importancia nativa posterior al preprocesamiento. Es una aproximacion: en presencia de muchas dummies correlacionadas, interpretar por grupos mas que por columnas individuales.

In [ ]:
rf_keys = [key for key in fitted if key[2] == 'RandomForest']
if rf_keys:
    rf_key = sorted(rf_keys, key=lambda k: results[(results['dataset'].eq(k[0])) & (results['source_variant'].eq(k[1])) & (results['algorithm'].eq(k[2]))]['rmse'].iloc[0])[0]
    pipe = fitted[rf_key]
    prep = pipe.named_steps['prep']
    model = pipe.named_steps['model']
    try:
        names = prep.get_feature_names_out()
        imp = pd.DataFrame({'feature': names, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
        display(imp.head(40))
        imp.head(25).sort_values('importance').plot(kind='barh', x='feature', y='importance', figsize=(10, 8), legend=False)
        plt.title(f'Importancia RandomForest: {rf_key}')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'No se pudo extraer importancia: {exc}')

## 7. Conclusiones ejecutivas

- Comparar RMSE/MAE/R2 entre M1, M2 y M3 indica si los scores militares y el texto aportan ganancia incremental.
- La variante sin `source` es fundamental para auditar si el modelo depende de sesgo de cobertura.
- Si el rendimiento en 2026 es pobre, no necesariamente invalida el enfoque: puede indicar shift fuente-tiempo entre ACLED historico e IranWarLive/GDELT 2026.
- Si M3 mejora sobre M2, el texto contiene informacion operacional no capturada por reglas de arma/objetivo.
- En una siguiente iteracion conviene entrenar un modelo en dos etapas: letalidad binaria y severidad condicional.